In [15]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [23]:
BASE = '/content/'

# EirGrid files — 15-minute interval data
DEMAND_FILES = [
    BASE + 'ROI_demandactual_22_Eirgrid.csv',
    BASE + 'ROI_demandactual_23_Eirgrid.csv',
    BASE + 'ROI_demandactual_24_Eirgrid.csv',   # only up to Feb 19 2024
]
CO2_EMISSION_FILES = [
    BASE + 'ROI_co2emission_22_Eirgrid.csv',
    BASE + 'ROI_co2emission_23_Eirgrid.csv',
    BASE + 'ROI_co2emission_24_Eirgrid.csv',
]
CO2_INTENSITY_FILES = [
    BASE + 'ROI_co2intensity_22_Eirgrid.csv',
    BASE + 'ROI_co2intensity_23_Eirgrid.csv',
    BASE + 'ROI_co2intensity_24_Eirgrid.csv',
]
WIND_FILES = [
    BASE + 'ROI_windactual_22_Eirgrid.csv',
    BASE + 'ROI_windactual_23_Eirgrid.csv',
    BASE + 'ROI_windactual_24_Eirgrid.csv',
]

# ENTSO-E full 2024 demand — covers the gap in EirGrid 2024 file
ENTSO_FILE = BASE + 'ENTSO_demand_2024_full.csv'

# Weather, events, holidays
WEATHER_FILE  = BASE + 'open-meteo-53.46N6.18W65m.csv'
EVENTS_FILE   = BASE + 'dublin_events_2022_2024_final.csv'
HOLIDAY_FILES = [BASE + '2022.csv', BASE + '2023.csv', BASE + '2024.csv']

print("File paths set")

File paths set


In [24]:
# Load and combine raw EirGrid CSV files (demand, CO2, wind)
def load_eirgrid(files, value_col_name):
    dfs = []
    for f in files:
        df = pd.read_csv(f, header=None,
                         names=['datetime', 'metric', 'region', 'value'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    combined['datetime'] = pd.to_datetime(combined['datetime'],
                                           format='%d-%b-%Y %H:%M:%S')
    combined = combined.drop_duplicates(subset=['datetime'])
    combined = combined.dropna(subset=['value'])
    combined = (combined[['datetime', 'value']]
                .rename(columns={'value': value_col_name})
                .set_index('datetime')
                .sort_index())
    return combined

demand_15min  = load_eirgrid(DEMAND_FILES,        'demand_mw')
co2_emission  = load_eirgrid(CO2_EMISSION_FILES,  'co2_emission')
co2_intensity = load_eirgrid(CO2_INTENSITY_FILES, 'co2_intensity')
wind_actual   = load_eirgrid(WIND_FILES,          'wind_mw')

print(f"Demand loaded:      {len(demand_15min):,} rows | {demand_15min.index.min().date()} → {demand_15min.index.max().date()}")
print(f"CO2 emission loaded:{len(co2_emission):,} rows | {co2_emission.index.min().date()} → {co2_emission.index.max().date()}")
print(f"CO2 intensity loaded:{len(co2_intensity):,} rows | {co2_intensity.index.min().date()} → {co2_intensity.index.max().date()}")
print(f"Wind loaded:        {len(wind_actual):,} rows | {wind_actual.index.min().date()} → {wind_actual.index.max().date()}")

Demand loaded:      74,780 rows | 2022-01-01 → 2024-02-19
CO2 emission loaded:71,178 rows | 2022-01-01 → 2024-02-19
CO2 intensity loaded:74,775 rows | 2022-01-01 → 2024-02-19
Wind loaded:        74,780 rows | 2022-01-01 → 2024-02-19


In [25]:
# Load ENTSO-E load data to fill the EirGrid 2024 demand gap
entso_raw = pd.read_csv(ENTSO_FILE)

# Parse datetime — ENTSO-E format is "01/01/2024 00:00 - 01/01/2024 00:30"
# We take just the start of each interval
entso_raw['datetime'] = (entso_raw['MTU (UTC)']
                          .str.split(' - ')
                          .str[0])
entso_raw['datetime'] = pd.to_datetime(entso_raw['datetime'],
                                        format='%d/%m/%Y %H:%M')
entso_raw['demand_mw'] = pd.to_numeric(
    entso_raw['Actual Total Load (MW)'], errors='coerce'
)

entso_demand = (entso_raw[['datetime', 'demand_mw']]
                .dropna(subset=['demand_mw'])
                .set_index('datetime')
                .sort_index())

print(f"ENTSO-E demand loaded: {len(entso_demand):,} rows | {entso_demand.index.min().date()} → {entso_demand.index.max().date()}")
print(f"   Missing values: {entso_demand['demand_mw'].isna().sum()}")

ENTSO-E demand loaded: 17,434 rows | 2024-01-01 → 2024-12-31
   Missing values: 0


In [26]:
# Resample CO2 and wind series from 15-min to 30-min
demand_eirgrid_30 = demand_15min.resample('30min').mean()

# Step 1 — reindex ENTSO-E to match the full 30-min timeline
full_index = pd.date_range(
    start='2022-01-01',
    end='2024-12-31 23:30',
    freq='30min'
)
demand_combined = demand_eirgrid_30.reindex(full_index)

# Step 2 — fill missing slots using ENTSO-E values
entso_aligned = entso_demand.reindex(full_index)
demand_combined['demand_mw'] = demand_combined['demand_mw'].fillna(
    entso_aligned['demand_mw']
)

print(f"Combined demand series: {len(demand_combined):,} rows")
print(f"   Range: {demand_combined.index.min().date()} → {demand_combined.index.max().date()}")
print(f"   Remaining NaN: {demand_combined['demand_mw'].isna().sum()}")
print(f"\n   EirGrid contributed: Jan 2022 → Feb 19 2024")
print(f"   ENTSO-E contributed: Feb 20 2024 → Dec 31 2024")

Combined demand series: 52,608 rows
   Range: 2022-01-01 → 2024-12-31
   Remaining NaN: 147

   EirGrid contributed: Jan 2022 → Feb 19 2024
   ENTSO-E contributed: Feb 20 2024 → Dec 31 2024


In [27]:
# Build the base EirGrid dataframe by joining CO2 and wind onto demand
co2_em_30  = co2_emission.resample('30min').mean()
co2_int_30 = co2_intensity.resample('30min').mean()
wind_30    = wind_actual.resample('30min').mean()

print(f"CO2 emission resampled:  {len(co2_em_30):,} rows | ends {co2_em_30.index.max().date()}")
print(f"CO2 intensity resampled: {len(co2_int_30):,} rows | ends {co2_int_30.index.max().date()}")
print(f"Wind resampled:          {len(wind_30):,} rows | ends {wind_30.index.max().date()}")

CO2 emission resampled:  37,423 rows | ends 2024-02-19
CO2 intensity resampled: 37,422 rows | ends 2024-02-19
Wind resampled:          37,423 rows | ends 2024-02-19


In [28]:
eirgrid = demand_combined.copy()
eirgrid = eirgrid.join([co2_em_30, co2_int_30, wind_30], how='left')

print(f"EirGrid base built: {eirgrid.shape}")
print(f"   Range: {eirgrid.index.min().date()} → {eirgrid.index.max().date()}")
print(f"\n   NaN counts per column:")
print(eirgrid.isnull().sum())

EirGrid base built: (52608, 4)
   Range: 2022-01-01 → 2024-12-31

   NaN counts per column:
demand_mw          147
co2_emission     16909
co2_intensity    15216
wind_mw          15214
dtype: int64


In [29]:
# Load and clean the Open-Meteo weather data
weather_raw = pd.read_csv(WEATHER_FILE, skiprows=3)
weather_raw.columns = ['datetime', 'temp_c', 'precip_mm',
                        'wind_speed_kmh', 'humidity_pct',
                        'dewpoint_c', 'surface_pressure_hpa']
weather_raw['datetime'] = pd.to_datetime(weather_raw['datetime'])
weather_raw = weather_raw.set_index('datetime').sort_index()

weather_30 = weather_raw.resample('30min').ffill()

print(f"Weather loaded: {len(weather_30):,} rows | {weather_30.index.min().date()} → {weather_30.index.max().date()}")

Weather loaded: 52,607 rows | 2022-01-01 → 2024-12-31


In [30]:
# Load and aggregate the Dublin events catalogue to daily level
events_raw = pd.read_csv(EVENTS_FILE)
events_raw['date'] = pd.to_datetime(events_raw['date'], format='%d-%m-%Y')

events_daily = events_raw.groupby('date').agg(
    event_flag      = ('event_flag', 'max'),
    event_intensity = ('intensity',  'max'),
    num_events      = ('event_name', 'count'),
).reset_index().set_index('date')

print(f"Events loaded: {len(events_raw)} records → {len(events_daily)} unique event days")
print(f"   Range: {events_daily.index.min().date()} → {events_daily.index.max().date()}")

Events loaded: 65 records → 63 unique event days
   Range: 2022-02-05 → 2024-10-27


In [31]:
# Load and combine the public holiday CSV files
holidays_raw = pd.concat(
    [pd.read_csv(f) for f in HOLIDAY_FILES],
    ignore_index=True
)
holidays_raw['date'] = pd.to_datetime(holidays_raw['date'],
                                       format='%d-%m-%Y',
                                       errors='coerce')
holidays_raw = holidays_raw.dropna(subset=['date'])

# Keep only actual Public Holidays — not observances
public_holidays = holidays_raw[holidays_raw['type'] == 'Public holiday']
holiday_days = (public_holidays[['date']]
                .drop_duplicates()
                .assign(is_public_holiday=1)
                .set_index('date'))

print(f" Public holidays loaded: {len(holiday_days)} dates across 2022–2024")

 Public holidays loaded: 31 dates across 2022–2024


In [32]:
# Merge weather, events, and holidays onto the base EirGrid data
master = eirgrid.copy()
master = master.join(weather_30, how='left')

# Events and holidays are daily — merge on date
master['date'] = master.index.normalize()
master = master.reset_index()  # datetime becomes a column

master = master.merge(events_daily.reset_index(), on='date', how='left')
master = master.merge(holiday_days.reset_index(), on='date', how='left')

# Set index back — use whichever column name exists
if 'datetime' in master.columns:
    master = master.set_index('datetime')
elif 'index' in master.columns:
    master = master.set_index('index')
    master.index.name = 'datetime'

master = master.drop(columns=['date'], errors='ignore')

# Fill NaN — 0 means no event / not a holiday
master['event_flag']        = master['event_flag'].fillna(0).astype(int)
master['event_intensity']   = master['event_intensity'].fillna(0)
master['num_events']        = master['num_events'].fillna(0).astype(int)
master['is_public_holiday'] = master['is_public_holiday'].fillna(0).astype(int)

print(f"Master merged: {master.shape}")
print(f"   Range: {master.index.min().date()} → {master.index.max().date()}")

Master merged: (52608, 14)
   Range: 2022-01-01 → 2024-12-31


In [33]:
# Engineer calendar/time features (hour, day of week, month, etc.)
master['hour']          = master.index.hour
master['minute']        = master.index.minute
master['day_of_week']   = master.index.dayofweek   # 0=Mon, 6=Sun
master['day_of_month']  = master.index.day
master['month']         = master.index.month
master['quarter']       = master.index.quarter
master['year']          = master.index.year
master['is_weekend']    = (master['day_of_week'] >= 5).astype(int)
master['halfhour_slot'] = master['hour'] * 2 + (master['minute'] // 30)

print("Time features added")

Time features added


In [34]:
# Engineer weather-derived features (temp squared, wind chill)
master['temp_squared'] = master['temp_c'] ** 2

def wind_chill(T, V):
    # Environment Canada formula — validated for I-SEM by Harkin & Liu 2024
    mask = (T <= 10) & (V > 4.8)
    return np.where(
        mask,
        13.12 + 0.6215*T - 11.37*(V**0.16) + 0.3965*T*(V**0.16),
        T
    )

master['wind_chill'] = wind_chill(
    master['temp_c'].values,
    master['wind_speed_kmh'].values
)
master['wind_penetration'] = np.where(
    master['demand_mw'] > 0,
    master['wind_mw'] / master['demand_mw'],
    0
)

print("Weather-derived features added")

Weather-derived features added


In [35]:
# Engineer cyclical sine/cosine encodings for hour, day, month
master['hour_sin']  = np.sin(2 * np.pi * master['hour'] / 24)
master['hour_cos']  = np.cos(2 * np.pi * master['hour'] / 24)
master['dow_sin']   = np.sin(2 * np.pi * master['day_of_week'] / 7)
master['dow_cos']   = np.cos(2 * np.pi * master['day_of_week'] / 7)
master['month_sin'] = np.sin(2 * np.pi * master['month'] / 12)
master['month_cos'] = np.cos(2 * np.pi * master['month'] / 12)

print("Cyclic encodings added")

Cyclic encodings added


In [38]:
# Engineer demand lag and rolling-mean features
master['demand_lag_24h']  = master['demand_mw'].shift(48)   # same slot yesterday
master['demand_lag_168h'] = master['demand_mw'].shift(336)  # same slot last week
master['demand_lag_1']    = master['demand_mw'].shift(1)    # previous half-hour
master['demand_rolling_24h_mean'] = (
    master['demand_mw'].shift(1).rolling(window=48, min_periods=24).mean()
)

print("Lag features added")
print(f"   Total columns so far: {master.shape[1]}")

Lag features added
   Total columns so far: 36


In [39]:
# Drop rows where demand is missing
master = master.dropna(subset=['demand_mw'])

# Forward-fill then back-fill weather gaps
weather_cols = ['temp_c', 'precip_mm', 'wind_speed_kmh', 'humidity_pct',
                'dewpoint_c', 'surface_pressure_hpa', 'temp_squared', 'wind_chill']
master[weather_cols] = master[weather_cols].ffill().bfill()

# Forward-fill CO2 and wind gaps (after Feb 19 2024)
master[['co2_emission', 'co2_intensity', 'wind_mw', 'wind_penetration']] = (
    master[['co2_emission', 'co2_intensity', 'wind_mw', 'wind_penetration']]
    .ffill().bfill()
)

print("Missing values handled")
print(f"\nRemaining NaN (lag cols only — by design):")
nulls = master.isnull().sum()
print(nulls[nulls > 0])

Missing values handled

Remaining NaN (lag cols only — by design):
demand_lag_24h              48
demand_lag_168h            336
demand_lag_1                 1
demand_rolling_24h_mean     24
dtype: int64


In [40]:
# Save the final merged dataset to CSV
OUTPUT = BASE + 'final_dataset_2022_2024.csv'
master.to_csv(OUTPUT)

print(f"Saved: final_dataset_2022_2024.csv")
print(f"\nShape:      {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"Date range: {master.index.min().date()} → {master.index.max().date()}")
print(f"\nDemand stats (MW):")
print(master['demand_mw'].describe().round(1))

event_days = master[master['event_flag']==1].resample('D').first()
print(f"\nEvent days in dataset:    {len(event_days)}")
print(f"Holiday slots in dataset: {master['is_public_holiday'].sum():,}")

avg_e = master[master['event_flag']==1]['demand_mw'].mean()
avg_n = master[master['event_flag']==0]['demand_mw'].mean()
print(f"\nAvg demand — event days:     {avg_e:.1f} MW")
print(f"Avg demand — non-event days: {avg_n:.1f} MW")


Saved: final_dataset_2022_2024.csv

Shape:      52,461 rows × 36 columns
Date range: 2022-01-01 → 2024-12-31

Demand stats (MW):
count    52461.0
mean      3715.6
std        599.8
min       2438.5
25%       3226.0
50%       3761.0
75%       4125.0
max       5687.6
Name: demand_mw, dtype: float64

Event days in dataset:    996
Holiday slots in dataset: 1,488

Avg demand — event days:     3404.7 MW
Avg demand — non-event days: 3734.6 MW


In [ ]:
# Download the final dataset from Colab
from google.colab import files
files.download('/content/final_dataset_2022_2024.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>